In [ ]:
# ==================================================================================================
# MIMIC-IV Cohort Exploratory Data Analysis (EDA)
# ==================================================================================================
#
# 목적
# - 모델링 이전 단계에서 코호트 정의와 데이터 구조의 타당성 검증
# - 환자/입원 단위 혼재 여부, 분포 이상, 결측 패턴 사전 확인

    # ==============================================================================================
    # 전체 EDA 리포트 생성 파이프라인
    # ==============================================================================================
    #
    # 역할
    # - 개별 EDA 단계를 순차적으로 실행
    # - Markdown 리포트 형태로 결과를 누적 기록
    #
    # 구성 흐름
    # 1. 데이터 기본 구조 확인 (레코드 수, 환자 수, 컬럼 정보)
    # 2. 기술 통계 및 분포 점검 (평균, 중앙값 등등)
    # 3. 결측 패턴 분석
    # 4. 범주형 (성별, ICU unit) / 수치형 변수 (나이, los) 구조 검증 
    # 5. 최소한의 상관 구조 확인
    # 6. 시각화를 통한 직관적 검증
    #
    # 출력물
    # - EDA_Report.md
    # - EDA_Visualizations.png
    # ==============================================================================================

# ============================================================================================

In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

from scipy import stats 
import warnings 
warnings.filterwarnings('ignore') 

# 한글 폰트 설정 
plt.rcParams['font.family'] = 'DejaVu Sans' 
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False


class CohortEDA:
    def __init__(self, csv_path: str):
        self.data = pd.read_csv(csv_path)
        self.report_lines = []

    def generate_full_report(self, output_path: str):
        self._write_header()
        self._basic_info()
        self._descriptive_stats()
        self._missing_values()
        self._categorical_analysis()
        self._numerical_analysis()
        self._correlation_analysis()
        self._sliding_window_and_clipping_eda()
        self._visualizations()

        with open(output_path, "w", encoding="utf-8") as f:
            f.write("\n".join(self.report_lines))

        print(f"Report saved to: {output_path}")

    # --------------------------------------------------
    # Section writers
    # --------------------------------------------------
    def _write_header(self):
        self.report_lines.extend(
            [
                "# MIMIC-IV Cohort Exploratory Data Analysis Report",
                "",
                f"**Analysis Date:** {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}",
                "",
                "---",
                "",
            ]
        )

    def _basic_info(self):
        self.report_lines.extend(
            [
                "## 1. Dataset Basic Information",
                "",
                f"- **Total Records:** {len(self.data):,}",
                f"- **Unique Patients:** {self.data['subject_id'].nunique():,}",
                f"- **Number of Features:** {len(self.data.columns)}",
                "",
                "### Column Names and Data Types",
                "```",
            ]
        )

        for col in self.data.columns:
            self.report_lines.append(f"{col:25s} : {self.data[col].dtype}")

        self.report_lines.extend(["```", ""])

    def _descriptive_stats(self):
        self.report_lines.extend(
            [
                "## 2. Descriptive Statistics",
                "",
                "### Numerical Variables",
                "",
                "```",
                self.data.select_dtypes(include=[np.number]).describe().to_string(),
                "```",
                "",
                "### Key Metrics",
                "",
                f"- **Average hospital stays per patient:** {len(self.data) / self.data['subject_id'].nunique():.2f}",
                f"- **Median age:** {self.data['anchor_age'].median():.1f} years",
                f"- **Median LOS:** {self.data['los'].median():.2f} days",
                "",
            ]
        )

    def _missing_values(self):
        self.report_lines.extend(["## 3. Missing Values Analysis", ""])

        missing = self.data.isnull().sum()
        pct = (missing / len(self.data) * 100).round(2)

        if missing.sum() == 0:
            self.report_lines.append("✅ **No missing values detected.**")
        else:
            self.report_lines.extend(
                [
                    "| Column | Missing Count | Percentage |",
                    "|--------|---------------|------------|",
                ]
            )
            for col in self.data.columns:
                if missing[col] > 0:
                    self.report_lines.append(
                        f"| {col} | {missing[col]:,} | {pct[col]:.2f}% |"
                    )
        self.report_lines.append("")

    def _categorical_analysis(self):
        self.report_lines.extend(
            [
                "## 4. Categorical Variables Analysis",
                "",
                "### Gender Distribution",
                "",
            ]
        )

        g = self.data.drop_duplicates("subject_id")["gender"].value_counts()
        gp = (g / g.sum() * 100).round(2)

        self.report_lines.extend(
            [
                "| Gender | Count | Percentage |",
                "|--------|-------|------------|",
            ]
        )

        for k in g.index:
            self.report_lines.append(f"| {k} | {g[k]:,} | {gp[k]:.2f}% |")

        self.report_lines.extend(["", "### ICU Unit Distribution", ""])

        u = self.data["first_careunit"].value_counts().head(10)
        up = (u / len(self.data) * 100).round(2)

        self.report_lines.extend(
            [
                "| ICU Unit | Count | Percentage |",
                "|----------|-------|------------|",
            ]
        )

        for k in u.index:
            self.report_lines.append(f"| {k[:40]} | {u[k]:,} | {up[k]:.2f}% |")

        self.report_lines.append("")

    def _numerical_analysis(self):
        self.report_lines.extend(
            [
                "## 5. Numerical Variables Analysis",
                "",
                "### Age Distribution",
                "",
            ]
        )

        age = self.data.drop_duplicates("subject_id")["anchor_age"]
        bins = pd.cut(
            age,
            [0, 30, 50, 65, 80, 100],
            labels=["<30", "30-50", "50-65", "65-80", "80+"],
        )

        dist = bins.value_counts().sort_index()

        self.report_lines.extend(
            [
                "| Age Group | Count | Percentage |",
                "|-----------|-------|------------|",
            ]
        )

        for k in dist.index:
            self.report_lines.append(
                f"| {k} | {dist[k]:,} | {dist[k] / len(age) * 100:.2f}% |"
            )

        self.report_lines.extend(["", "### Length of Stay (LOS) Distribution", ""])

        for q, v in self.data["los"].quantile(
            [0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
        ).items():
            self.report_lines.append(f"- {int(q * 100)}th percentile: {v:.2f} days")

        self.report_lines.append("")

    def _correlation_analysis(self):
        self.report_lines.extend(
            [
                "## 6. Correlation Analysis",
                "",
                "```",
                self.data[["anchor_age", "los"]].corr().to_string(),
                "```",
                "",
            ]
        )

    def _sliding_window_and_clipping_eda(self):
        los_q = self.data["los"].quantile([0.9, 0.95, 0.99])

        self.report_lines.extend(
            [
                "## 8. Sliding Window & Clipping Decision EDA",
                "",
                "LOS 및 사망 시점 분포를 통해 시간 기반(sliding window) 분석의 필요성을 확인하였다.",
                "",
                "### LOS Distribution",
                f"- 90th percentile: {los_q.loc[0.9]:.2f} days",
                f"- 95th percentile: {los_q.loc[0.95]:.2f} days",
                f"- 99th percentile: {los_q.loc[0.99]:.2f} days",
                "",
                "LOS는 값 자체를 clipping 하기보다는 슬라이딩 윈도우 생성 단계에서 상한을 두는 것이 타당하다.",
                "",
            ]
        )

    def _create_plots(self):
         """시각화 생성"""
         fig = plt.figure(figsize=(24, 12))

        # Plot 1: 시간대별 입원
        if 'in_hour' in self.data.columns:
            plt.subplot(2, 3, 1)
            hour_data = self.data.groupby('in_hour').size()
            plt.bar(hour_data.index, hour_data.values, alpha=0.7)
            plt.xlabel('Hour of Day')
            plt.ylabel('Number of Admissions')
            plt.title('Patient Admissions by Hour')
            plt.grid(axis='y', alpha=0.3)
        # Plot 2: Age vs LOS
        plt.subplot(2, 3, 2)
        sample_data = self.data.sample(min(5000, len(self.data)))
        plt.scatter(
            sample_data['anchor_age'],
            sample_data['los'],
            alpha=0.3,
            s=10
        )
        plt.xlabel('Age (years)')
        plt.ylabel('Length of Stay (days, log scale)')
        plt.yscale('log')
        plt.title('Age vs Length of Stay')
        plt.grid(alpha=0.3)

        # Plot 3: ICU Unit × Gender
        plt.subplot(2, 3, 3)
        top_units = self.data['first_careunit'].value_counts().head(8)
        unit_gender = (
            self.data[self.data['first_careunit'].isin(top_units.index)]
            .groupby(['first_careunit', 'gender'])
            .size()
            .unstack(fill_value=0)
        )
         unit_gender.plot(kind='bar', stacked=True, ax=plt.gca())
        plt.xlabel('ICU Unit')
        plt.ylabel('Number of Patients')
        plt.title('Top ICU Units by Gender')
        plt.xticks(rotation=45, ha='right')

        # Plot 4: Missing values heatmap
        plt.subplot(2, 3, 4)
        missing_matrix = self.data.isnull().astype(int)
        if missing_matrix.sum().sum() > 0:
            sns.heatmap(
                missing_matrix.iloc[:1000].T,
                cmap='YlOrRd',
                cbar=True
            )
            plt.title('Missing Values Pattern (First 1000 Rows)')
        else:
            plt.text(0.5, 0.5, 'No Missing Values', ha='center', va='center')
            plt.title('Missing Values Pattern')
        # Plot 5: LOS Distribution
        plt.subplot(2, 3, 5)
        plt.hist(self.data['los'], bins=100)
        plt.yscale('log')
        plt.xlabel('Length of Stay (days)')
        plt.ylabel('Frequency (log scale)')
        plt.title('LOS Distribution (Right-skewed)')

        plt.tight_layout()
        plt.savefig('EDA_Visualizations.png', dpi=150, bbox_inches='tight')
        plt.show()

        self.report_lines.append("![Visualizations](EDA_Visualizations.png)")
        self.report_lines.append("")

if __name__ == "__main__":
    eda = CohortEDA(
        "/home/oracle/Coding/wsl_projects/miniprj/data-pipeline/data/processed/cohort_base.csv"
    )
    eda.generate_full_report("00_EDA_MIMIC_Cohort_EDA_Report.md")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

# 경고 메시지 무시 (데이터 분석 시 시각화 라이브러리의 경고 방지)
warnings.filterwarnings("ignore")

# 시각화 한글 폰트 설정 (환경에 따라 'NanumGothic' 등으로 변경 가능)
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

class CohortEDA:
    def __init__(self, csv_path: str):
        """데이터를 로드하고 리포트 라인을 초기화합니다."""
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"파일을 찾을 수 없습니다: {csv_path}")
        self.data = pd.read_csv(csv_path)
        self.report_lines = []

    def generate_full_report(self, output_path: str):
        """모든 분석 단계를 실행하고 마크다운 리포트를 생성합니다."""
        self._write_header()                    # 1. 헤더 작성
        self._basic_info()                      # 2. 기본 정보 (레코드 수 등)
        self._descriptive_stats()               # 3. 기술 통계량
        self._missing_values()                  # 4. 결측치 분석
        self._categorical_analysis()            # 5. 범주형 변수 분석
        self._numerical_analysis()              # 6. 수치형 변수 분석
        self._correlation_analysis()            # 7. 상관관계 분석
        self._sliding_window_and_clipping_eda() # 8. 윈도우/클리핑 전략 분석
        self._create_plots()                    # 9. 시각화 및 이미지 삽입

        # 리포트 파일 저장
        with open(output_path, "w", encoding="utf-8") as f:
            f.write("\n".join(self.report_lines))

        print(f"✅ 리포트가 저장되었습니다: {output_path}")

    def _write_header(self):
        """리포트 제목 및 분석 일시를 작성합니다."""
        self.report_lines.extend([
            "# MIMIC-IV 코호트 탐색적 데이터 분석(EDA) 보고서",
            "",
            f"**분석 일시:** {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}",
            "",
            "---",
            ""
        ])

    def _basic_info(self):
        """데이터셋의 크기, 고유 환자 수, 데이터 타입을 분석합니다."""
        self.report_lines.extend([
            "## 1. 데이터셋 기본 정보",
            "",
            f"- **총 레코드 수:** {len(self.data):,}",
            f"- **고유 환자 수(subject_id):** {self.data['subject_id'].nunique():,}",
            f"- **피처(컬럼) 수:** {len(self.data.columns)}",
            "",
            "### 컬럼별 데이터 타입",
            "```"
        ])
        for col in self.data.columns:
            self.report_lines.append(f"{col:25s} : {self.data[col].dtype}")
        self.report_lines.extend(["```", ""])

    def _descriptive_stats(self):
        """수치형 데이터의 요약 통계와 주요 지표를 계산합니다."""
        self.report_lines.extend([
            "## 2. 기술 통계량",
            "",
            "### 수치형 변수 요약",
            "```",
            self.data.select_dtypes(include=[np.number]).describe().to_string(),
            "```",
            "",
            "### 주요 의료 지표",
            f"- **환자당 평균 입원 횟수:** {len(self.data) / self.data['subject_id'].nunique():.2f}",
            f"- **연령 중앙값:** {self.data['anchor_age'].median():.1f} 세",
            f"- **재원 기간(LOS) 중앙값:** {self.data['los'].median():.2f} 일",
            ""
        ])

    def _missing_values(self):
        """결측치 존재 여부와 비율을 분석합니다."""
        self.report_lines.extend(["## 3. 결측치 분석", ""])
        missing = self.data.isnull().sum()
        if missing.sum() == 0:
            self.report_lines.append("✅ **결측치가 발견되지 않았습니다.**")
        else:
            pct = (missing / len(self.data) * 100).round(2)
            self.report_lines.extend([
                "| 컬럼명 | 결측치 수 | 비율 |",
                "|:---|:---:|:---:|"
            ])
            for col in self.data.columns:
                if missing[col] > 0:
                    self.report_lines.append(f"| {col} | {missing[col]:,} | {pct[col]}% |")
        self.report_lines.append("")

    def _categorical_analysis(self):
        """성별 및 ICU 병동 분포를 분석합니다."""
        self.report_lines.extend(["## 4. 범주형 변수 분석", "", "### 성별 분포 (고유 환자 기준)", ""])
        
        # 중복 환자 제거 후 성별 집계
        g = self.data.drop_duplicates("subject_id")["gender"].value_counts()
        gp = (g / g.sum() * 100).round(2)

        self.report_lines.extend(["| 성별 | 인원 수 | 비율 |", "|:---:|:---:|:---:|"])
        for k in g.index:
            self.report_lines.append(f"| {k} | {g[k]:,} | {gp[k]}% |")

        self.report_lines.extend(["", "### ICU 병동(Unit) 분포", ""])
        u = self.data["first_careunit"].value_counts().head(10)
        up = (u / len(self.data) * 100).round(2)

        self.report_lines.extend(["| ICU 병동 | 환자 수 | 비율 |", "|:---|:---:|:---:|"])
        for k in u.index:
            self.report_lines.append(f"| {k} | {u[k]:,} | {up[k]}% |")
        self.report_lines.append("")

    def _numerical_analysis(self):
        """연령대별 분포와 LOS의 백분위수를 분석합니다."""
        self.report_lines.extend(["## 5. 수치형 변수 분석", "", "### 연령대별 분포", ""])
        
        age = self.data.drop_duplicates("subject_id")["anchor_age"]
        bins = pd.cut(age, [0, 30, 50, 65, 80, 100], labels=["<30", "30-50", "50-65", "65-80", "80+"])
        dist = bins.value_counts().sort_index()

        self.report_lines.extend(["| 연령 그룹 | 인원 수 | 비율 |", "|:---:|:---:|:---:|"])
        for k in dist.index:
            self.report_lines.append(f"| {k} | {dist[k]:,} | {dist[k]/len(age)*100:.2f}% |")

        self.report_lines.extend(["", "### 재원 기간(LOS) 상세 분포 (Percentiles)", ""])
        for q, v in self.data["los"].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).items():
            self.report_lines.append(f"- 상위 {int(q*100)}%: {v:.2f} 일")
        self.report_lines.append("")

    def _correlation_analysis(self):
        """주요 변수 간의 상관계수를 산출합니다."""
        self.report_lines.extend([
            "## 6. 상관관계 분석",
            "",
            "```",
            self.data[["anchor_age", "los"]].corr().to_string(),
            "```",
            ""
        ])

    def _sliding_window_and_clipping_eda(self):
        """전처리 전략(슬라이딩 윈도우 등)을 결정하기 위한 분석입니다."""
        los_q = self.data["los"].quantile([0.9, 0.95, 0.99])
        self.report_lines.extend([
            "## 7. 슬라이딩 윈도우 및 클리핑 결정 근거",
            "",
            "재원 기간(LOS) 분포를 통해 시간 기반 시퀀스 데이터 생성 시의 상한선을 검토합니다.",
            "",
            f"- 90% 환자의 LOS: {los_q.loc[0.9]:.2f} 일 이내",
            f"- 95% 환자의 LOS: {los_q.loc[0.95]:.2f} 일 이내",
            f"- 99% 환자의 LOS: {los_q.loc[0.99]:.2f} 일 이내",
            "",
            "> **결론:** LOS 이상치에 의한 모델 왜곡을 방지하기 위해, 슬라이딩 윈도우 생성 시 상위 5% 수준에서 시간축 상한(Clipping)을 두는 것을 권장합니다.",
            ""
        ])

    def _create_plots(self):
        """데이터를 시각화하여 이미지 파일로 저장하고 리포트에 포함합니다."""
        fig, axes = plt.subplots(2, 3, figsize=(20, 10))
        axes = axes.flatten()

        # 1. 시간대별 입원 현황
        if 'in_hour' in self.data.columns:
            hour_data = self.data.groupby('in_hour').size()
            axes[0].bar(hour_data.index, hour_data.values, color='steelblue')
            axes[0].set_title('Hour of Admission')

        # 2. 연령 vs 재원기간 (산점도)
        sample = self.data.sample(min(3000, len(self.data)))
        axes[1].scatter(sample['anchor_age'], sample['los'], alpha=0.2)
        axes[1].set_yscale('log')
        axes[1].set_title('Age vs LOS (Log Scale)')

        # 3. 병동별 성별 분포 (누적 막대 그래프)
        top_units = self.data['first_careunit'].value_counts().head(5).index
        unit_gender = self.data[self.data['first_careunit'].isin(top_units)].groupby(['first_careunit', 'gender']).size().unstack()
        unit_gender.plot(kind='bar', stacked=True, ax=axes[2])
        axes[2].set_title('Top 5 Units by Gender')
        axes[2].tick_params(axis='x', rotation=45)

        # 4. 결측치 패턴 히트맵
        sns.heatmap(self.data.iloc[:500].isnull(), cbar=False, cmap='magma', ax=axes[3])
        axes[3].set_title('Missing Value Pattern')

        # 5. LOS 분포 (히스토그램)
        sns.histplot(self.data['los'], bins=50, kde=True, ax=axes[4])
        axes[4].set_yscale('log')
        axes[4].set_title('LOS Distribution')

        axes[5].axis('off') # 빈 공간 제거

        plt.tight_layout()
        img_name = "EDA_Visualizations.png"
        plt.savefig(img_name, dpi=120)
        plt.close()

        self.report_lines.extend(["## 8. 시각화 자료", f"![EDA Plots]({img_name})", ""])

if __name__ == "__main__":
    # 파일 경로 설정 (본인의 환경에 맞게 수정)
    FILE_PATH = "/home/oracle/Coding/wsl_projects/miniprj/data-pipeline/data/processed/cohort_base.csv"
    
    eda = CohortEDA(FILE_PATH)
    eda.generate_full_report("MIMIC_Cohort_EDA_Report.md")

In [ ]:
import pandas as pd
import os

def generate_exact_text_report(csv_path, output_name="MIMIC_EDA_Clean_Report.html"):
    # 데이터 로드
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"파일 로드 오류: {e}")
        return

    # 1. 결측치 테이블 생성 (AttributeError 수정 완료)
    missing_data = df.isnull().sum()
    missing_table_rows = ""
    for col, count in missing_data.items():
        if count > 0:
            # float 객체 에러 방지를 위해 내장 함수 round() 사용
            pct = round((count / len(df) * 100), 2)
            missing_table_rows += f"<tr><td>{col}</td><td>{count:,}</td><td>{pct}%</td></tr>"

    # --- HTML 템플릿 (보내주신 텍스트 및 구조 100% 유지, 시각화만 제외) ---
    html_content = f"""
    <!DOCTYPE html>
    <html lang="ko">
    <head>
        <meta charset="UTF-8">
        <style>
            body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; line-height: 1.6; color: #333; max-width: 900px; margin: 40px auto; padding: 20px; }}
            h1 {{ font-size: 26px; border-bottom: 2px solid #333; padding-bottom: 10px; margin-bottom: 20px; }}
            h2 {{ font-size: 20px; margin-top: 35px; border-left: 5px solid #333; padding: 8px 12px; background: #f9f9f9; }}
            h3 {{ font-size: 17px; margin-top: 25px; border-bottom: 1px solid #eee; padding-bottom: 5px; }}
            pre {{ background: #f4f4f4; padding: 15px; border: 1px solid #ddd; overflow-x: auto; font-family: 'Cascadia Code', 'Courier New', monospace; font-size: 13px; border-radius: 4px; }}
            table {{ width: 100%; border-collapse: collapse; margin: 20px 0; font-size: 14px; }}
            th, td {{ border: 1px solid #ddd; padding: 10px; text-align: left; }}
            th {{ background: #f2f2f2; font-weight: bold; }}
            .text-block {{ margin: 15px 0; white-space: pre-wrap; }}
            ul {{ margin: 15px 0; padding-left: 20px; }}
            li {{ margin-bottom: 5px; }}
            hr {{ border: 0; border-top: 1px solid #eee; margin: 20px 0; }}
        </style>
    </head>
    <body>
        <h1># MIMIC-IV Cohort Exploratory Data Analysis Report</h1>
        <p><strong>Analysis Date:</strong> 2026-01-09 22:00:13</p>
        <hr>

        <h2>1. Dataset Basic Information</h2>
        <ul>
            <li><strong>Total Records:</strong> 54,551</li>
            <li><strong>Unique Patients:</strong> 54,551</li>
            <li><strong>Number of Features:</strong> 20</li>
        </ul>

        <h3>Column Names and Data Types</h3>
        <pre>{df.dtypes.to_string()}</pre>

        <div class="text-block">
본 코호트는 54,551개의 ICU 입실(stay)으로 구성되며, 환자 수와 record 수가 동일하다.
이는 본 데이터가 환자당 1개의 ICU stay만 포함하도록 이미 정제된 코호트임을 의미한다.

이러한 구조는 이후 조기 사망 예측이나 중재 필요 알림과 같이 시간에 따라 상태가 변화하는 문제를 다루기 위해서 슬라이딩 윈도우 적용 시
각 stay를 기준으로 시간 축 방향으로 샘플을 확장하기에 적합하다.

총 20개의 변수는 인구학적 정보, 입실·퇴실 시점, 사망 여부,
그리고 주요 중재 이벤트 시작 시점을 포함하고 있으며,
조기 악화 예측을 위한 메타 코호트 테이블로서 충분한 정보를 제공한다.
        </div>

        <h2>2. Descriptive Statistics</h2>
        <h3>Numerical Variables</h3>
        <pre>{df.describe().to_string()}</pre>

        <div class="text-block">
중앙 연령은 65세, 중앙 LOS는 2.37일로,
대부분의 환자는 비교적 짧은 ICU 재실 기간을 가지지만
일부 장기 입원 환자가 평균 LOS를 크게 증가시키는 우측 편향 분포를 보인다.

사망 관련 지표를 보면 hospital mortality는 약 10.8%로
조기 사망 예측 문제에서 명확한 class imbalance가 존재함을 확인할 수 있다.
이는 이후 모델링 단계에서 PR-AUC, recall 중심 평가가 필요함을 시사한다.

전처리 관점에서 LOS는 feature로 직접 사용하지 않되,
슬라이딩 윈도우 생성 시 샘플 수를 제어하는 기준 변수로 활용하는 것이 적절하다.
        </div>

        <h3>Key Metrics</h3>
        <ul>
            <li><strong>Average hospital stays per patient:</strong> 1.00</li>
            <li><strong>Median age:</strong> 65.0 years</li>
            <li><strong>Median LOS:</strong> 2.37 days</li>
        </ul>

        <h2>3. Missing Values Analysis</h2>
        <table>
            <thead><tr><th>Column</th><th>Missing Count</th><th>Percentage</th></tr></thead>
            <tbody>
                {missing_table_rows}
            </tbody>
        </table>

        <div class="text-block">
사망 시점(deathtime, dod)과 중재 시작 시점(vent_start_time, pressor_start_time)에서
높은 결측률이 관찰되지만, 이는 데이터 품질 문제라기보다
사망 또는 중재가 발생하지 않은 환자가 다수 존재하기 때문인 구조적 결측이다.

특히 vent_start_time, pressor_start_time의 높은 결측률은
해당 중재가 모든 환자에게 적용되지 않는 이벤트 기반 변수임을 의미한다.

따라서 이들 변수는 단순 결측 보간 대상이 아니라, 이벤트 발생 여부 및 시점 정보로 재해석되어야 한다.

따라서 전처리 단계에서는 다음과 같은 방향이 제시될 수 있다.
- 사망 시점 변수 → label 생성 전용
- 중재 시점 변수 → 이벤트 발생 여부 + 시점 기반 feature
- 결측 자체를 “미발생” 상태로 해석

이는 이후 중재 필요 알림(ventilator/pressor) 예측을 위한 핵심 전처리 설계로 이어질 수 있다.
        </div>

        <h2>4. Categorical Variables Analysis</h2>
        <h3>Gender Distribution</h3>
        <table>
            <tr><th>Gender</th><th>Count</th><th>Percentage</th></tr>
            <tr><td>M</td><td>31,056</td><td>56.93%</td></tr>
            <tr><td>F</td><td>23,495</td><td>43.07%</td></tr>
        </table>

        <div class="text-block">
성별 분포는 비교적 균형을 이루고 있으며, 남성(56.9%)이 여성(43.1%)보다 다소 높은 비율을 보인다.
ICU unit 분포를 보면 주요 중환자실 유형이 고르게 포함되어 있어 범용적인 ICU 조기 악화 신호를 학습할 가능성이 있다.
        </div>

        <h2>5. Sliding Window & Clipping Strategy</h2>
        <h3>LOS Distribution Summary</h3>
        <ul>
            <li>90th percentile: 8.99 days</li>
            <li>95th percentile: 13.89 days</li>
            <li>99th percentile: 28.09 days</li>
        </ul>

        <div class="text-block">
ICU 재실 기간(LOS)은 본질적으로 우측으로 긴 꼬리를 가지는 분포를 보이며, 대부분의 환자는 1–3일 내에 ICU를 퇴실하지만 일부 중증 환자는 다장기 부전, 반복적 중재, 합병증 또는 회복 지연으로 인해 수 주 이상 장기간 재실하게 된다.

다만 LOS는 환자의 퇴실 이후에야 확정되는 미래 정보이므로 예측 모델의 입력 feature로 사용하는 것은 데이터 누수를 유발할 수 있으며, 대신 슬라이딩 윈도우 기반 전처리 과정에서 구조적 제어 변수로 활용하는 것이 타당하다.

이에 본 프로젝트에서는 환자를 분석 대상에서 제외하지 않고, 슬라이딩 윈도우 생성 단계에서만 최대 관찰 기간에 상한(Clipping)을 두어 환자별 평가 시점 수를 제한하는 방식을 채택한다. 본 프로젝트에서는 1시간 단위 stride와 과거 6시간 관찰 window를 적용하며, LOS는 윈도우 생성 단계에서만 상한을 두는 방식으로 관리한다.
        </div>
    </body>
    </html>
    """

    with open(output_name, "w", encoding="utf-8") as f:
        f.write(html_content)
    print(f"✅ 리포트가 성공적으로 생성되었습니다: {os.path.abspath(output_name)}")

# 실행
CSV_PATH = "/home/oracle/Coding/wsl_projects/miniprj/data-pipeline/data/processed/cohort_base.csv"
generate_exact_text_report(CSV_PATH)

In [ ]:
#